# Random forest



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=False)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Gli alberi richiedono un compromesso: un albero profondo con tante foglie può sovradattarsi perché ogni previsione dipende da poche case; uno troppo semplice ignora differenze importanti.

Anche tecniche più sofisticate affrontano questo problema. Una soluzione efficace è la **random forest**, una foresta di alberi casuali. Nella regressione la previsione è la media delle previsioni degli alberi. Spesso funziona meglio di un albero singolo anche con i parametri predefiniti. Esistono altri modelli molto efficaci, ma alcuni richiedono una regolazione più attenta.

# Esempio
Dopo il caricamento e la divisione dei dati disponiamo di:

* `train_X`
* `val_X`
* `train_y`
* `val_y`


In [ ]:
import pandas as pd
    
# Carica i dati.
melbourne_file_path = '../input/melbourne-housing-snapshot/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 
# Escludi le righe con valori mancanti.
melbourne_data = melbourne_data.dropna(axis=0)
# Scegli il target e le feature.
y = melbourne_data.Price
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 
                        'YearBuilt', 'Lattitude', 'Longtitude']
X = melbourne_data[melbourne_features]

from sklearn.model_selection import train_test_split

# Dividi feature e target nei set di training e validazione.
# La suddivisione usa un generatore di numeri casuali. Assegnando un numero
# all'argomento random_state si ottiene la stessa suddivisione ogni volta che si
# eseguire nuovamente lo script.
train_X, val_X, train_y, val_y = train_test_split(X, y,random_state = 0)

La costruzione è simile a quella dell'albero, ma usiamo **`RandomForestRegressor`** al posto di `DecisionTreeRegressor`.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

forest_model = RandomForestRegressor(random_state=1)
forest_model.fit(train_X, train_y)
melb_preds = forest_model.predict(val_X)
print(mean_absolute_error(val_y, melb_preds))

# Riepilogo
Nell'esempio la random forest migliora sensibilmente rispetto al MAE di circa 250.000 dell'albero. Possiamo modificarne i parametri come abbiamo fatto per il numero di foglie, ma un vantaggio è che spesso funziona già bene senza una lunga ricerca.

Prova una **[random forest nell'esercizio](https://prof-davidaffo.github.io/kaggle-learntools/corsi/machine_learning/ex6.html)** e confronta i risultati.
